# Modeling - EfficientNet-B0

## Import

In [17]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

In [18]:
INPUT_ROOT = Path("/kaggle/input")

# 1. Mencari repository
repo_candidates = list(
    INPUT_ROOT.rglob("scripts/config.py")
)

if not repo_candidates:
    raise FileNotFoundError(
        "Repository belum ditambahkan sebagai Kaggle Input."
    )

REPO_ROOT = repo_candidates[0].parent.parent

# 2. Mencari dataset kompetisi
csv_candidates = list(INPUT_ROOT.rglob("train.csv"))

csv_candidates = [
    p for p in csv_candidates
    if (p.parent / "test.csv").exists()
    and (p.parent / "sample_submission.csv").exists()
]

if not csv_candidates:
    raise FileNotFoundError(
        "Dataset kompetisi tidak ditemukan."
    )

DATA_ROOT = csv_candidates[0].parent

# 3. Mencari pretrained weights
weight_candidates = list(
    INPUT_ROOT.rglob(
        "efficientnet_b0_rwightman-7f5810bc.pth"
    )
)

if not weight_candidates:
    raise FileNotFoundError(
        "Pretrained EfficientNet-B0 tidak ditemukan."
    )

WEIGHTS_PATH = weight_candidates[0]

# 4. Mengatur direktori
os.environ["DOC_DATA_DIR"] = str(DATA_ROOT)
os.environ["DOC_CHECKPOINT_DIR"] = "/kaggle/working/checkpoint"
os.environ["DOC_SUBMISSION_DIR"] = "/kaggle/working/submissions"
os.environ["DOC_REPORT_DIR"] = "/kaggle/working/reports"
os.environ["DOC_WEIGHTS_PATH"] = str(WEIGHTS_PATH)

# 5. Menghubungkan repository
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# 6. Import konfigurasi
from scripts.config import EXPERIMENTS

# 7. Menampilkan hasil
print("Repository:", REPO_ROOT)
print("Dataset:", DATA_ROOT)
print("Pretrained weights:", WEIGHTS_PATH)
print("Ukuran weights:", round(WEIGHTS_PATH.stat().st_size / 1024**2, 2), "MB")
print("Output: /kaggle/working")

print("\nEksperimen:")
for name, config in EXPERIMENTS.items():
    print(name, ":", config)

Repository: /kaggle/input/datasets/ameliawidia/document-image-classification/document-image-classification/kaggle-document-classification-repo
Dataset: /kaggle/input/competitions/document-image-classification-sd234504
Pretrained weights: /kaggle/input/datasets/ameliawidia/efficientnet-b0-pretrained/efficientnet_b0_rwightman-7f5810bc.pth
Ukuran weights: 20.45 MB
Output: /kaggle/working

Eksperimen:
baseline : {'display_name': 'Baseline', 'dropout': 0.3, 'backbone_lr': 0.0001, 'head_lr': 0.0008, 'weight_decay': 0.0001, 'epochs': 10, 'checkpoint': 'best_document_efficientnet_b0_baseline.pth', 'history': 'baseline_history.json'}
tuning1 : {'display_name': 'Hyperparameter Tuning 1', 'dropout': 0.4, 'backbone_lr': 5e-05, 'head_lr': 0.0003, 'weight_decay': 0.0005, 'epochs': 12, 'checkpoint': 'best_document_efficientnet_b0_tuning.pth', 'history': 'tuning1_history.json'}
tuning2 : {'display_name': 'Hyperparameter Tuning 2', 'dropout': 0.35, 'backbone_lr': 7.5e-05, 'head_lr': 0.0005, 'weight_dec

## Training 

### Trining 1: baseline

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m", "scripts.train",
        "--experiment", "baseline",
        "--data-dir", str(DATA_ROOT),
        "--weights", str(WEIGHTS_PATH),
        "--batch-size", "16",
        "--num-workers", "2"
    ],
    check=True
)

### Training 2: Hyperparameter Tuning 1

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m", "scripts.train",
        "--experiment", "tuning1",
        "--data-dir", str(DATA_ROOT),
        "--weights", str(WEIGHTS_PATH),
        "--batch-size", "16",
        "--num-workers", "2"
    ],
    check=True
)

### Training 3: Hyperparameter Tuning 2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m", "scripts.train",
        "--experiment", "tuning2",
        "--data-dir", str(DATA_ROOT),
        "--weights", str(WEIGHTS_PATH),
        "--batch-size", "16",
        "--num-workers", "2"
    ],
    check=True
)

## Evaluasi ketiga eksperimen

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m", "scripts.evaluate",
        "--experiment", "all",
        "--data-dir", str(DATA_ROOT),
        "--batch-size", "16",
        "--num-workers", "2"
    ],
    check=True
)

## Visualisasi hasil perbandingan

In [ ]:
comparison_path = Path("/kaggle/working/reports/figures/experiment_comparison.csv")
comparison = pd.read_csv(comparison_path)
display(
    comparison.sort_values(
        "Validation Accuracy",
        ascending=False
    )
)

In [19]:
import shutil
from pathlib import Path

checkpoint_dir = Path("/kaggle/working/checkpoint")

shutil.make_archive(
    "/kaggle/working/checkpoint_models",
    "zip",
    root_dir=checkpoint_dir
)

print("ZIP berhasil dibuat!")

ZIP berhasil dibuat!
